<a href="https://colab.research.google.com/github/your-org/alexpose/blob/main/experiments/multiple-sclerosis/03_sjepa_model_and_pretrain_normal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 - Build S-JEPA and pretrain on training sources

S-JEPA predicts hidden skeleton features. Its view encoder reads visible joints, its predictor receives joint and time positions, and its target encoder supplies targets. The target encoder is a slow moving average of the view encoder.

The filename is historical: we train on all three conditions in the training partition, with no condition labels in the loss. Unlabeled test motion would still leak information, so neither validation nor test windows enter training. Run notebook 01 first.

In [ ]:
# --- Setup: install dependencies (Colab installs; local usually already has them) ---
import importlib, importlib.util, subprocess, sys, os

IN_COLAB = 'google.colab' in sys.modules

def _need(mod):
    return importlib.util.find_spec(mod) is None

# Light deps used by every notebook.
_pkgs = []
for mod, pip_name in [('cv2','opencv-python'), ('mediapipe','mediapipe'),
                      ('sklearn','scikit-learn'), ('pandas','pandas'),
                      ('matplotlib','matplotlib'), ('tqdm','tqdm')]:
    if _need(mod):
        _pkgs.append(pip_name)
# torch is guarded so Colab's preinstalled GPU torch is never downgraded.
if _need('torch'):
    _pkgs.append('torch')
if _pkgs:
    print('installing:', _pkgs)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_pkgs])
else:
    print('all light dependencies already present')

In [ ]:
# --- Make `sjepa` and `ambient` importable, locally and in Colab ---
from pathlib import Path
import sys, subprocess

def _find_exp_dir():
    # Local run: this notebook sits in experiments/multiple-sclerosis.
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / 'sjepa' / '__init__.py').exists():
            return p
    return None

EXP_DIR = _find_exp_dir()
if EXP_DIR is None:
    # Colab: clone the repo, then point at the experiment folder.
    REPO = 'https://github.com/your-org/alexpose.git'  # <-- edit to your fork
    if not Path('alexpose').exists():
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO])
    EXP_DIR = Path('alexpose') / 'experiments' / 'multiple-sclerosis'

REPO_ROOT = EXP_DIR.parents[1]
for p in (str(EXP_DIR), str(REPO_ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
print('experiment dir:', EXP_DIR)
print('repo root     :', REPO_ROOT)

In [ ]:
# --- Paths and profile (reads the root .env if python-dotenv is present) ---
import os
try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / '.env')
except Exception:
    pass

VIDEO_DIR = EXP_DIR / 'video-data-full'
ARTIFACT_DIR = EXP_DIR / 'artifacts'
KEYPOINTS_DIR = ARTIFACT_DIR / 'keypoints-full'
IMAGES_DIR = EXP_DIR / 'images'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Pick the model size profile. 'laptop' is the fast default; set SJEPA_PROFILE=gpu
# in your .env for a larger model, or SJEPA_SMOKE=1 for a near-instant test run.
os.environ.setdefault('SJEPA_PROFILE', 'laptop')
print('SJEPA_PROFILE =', os.environ['SJEPA_PROFILE'],
      '| SJEPA_SMOKE =', os.environ.get('SJEPA_SMOKE', '0'))

## Keep each source video together

We use the same frozen five-fold registry in notebooks 02–06. A source video may have several clips; each clip may yield overlapping windows. All of those relatives stay together. Each round uses about 60% of sources for training, 20% for validation, and 20% for testing. Only notebook 06 evaluates test clips.

The splitter runs on one row per source, with condition labels used to balance source counts. It never splits windows. The loader checks the full cache, reviewed exclusions, and registry checksum. A changed cache requires a new registry and new checkpoints. See [the full method](docs/11-full-data-splits.md).

In [ ]:
from IPython.display import display
import pandas as pd
from sjepa.splits import load_full_registry, partition_records, split_summary
records, registry = load_full_registry(EXP_DIR)
FOLD = 0  # teaching example; notebook 06 independently trains all five folds
train_recs, val_recs, test_recs = partition_records(records, registry, FOLD)
display(pd.DataFrame(split_summary(records, registry)))
print('usable clips:', len(records), '| excluded raw clips:', len(registry['inventory']['exclusions']))
print('registry:', registry['registry_sha256'])

In [ ]:
from IPython.display import SVG, display
display(SVG(filename=str(IMAGES_DIR / 'sjepa_two_lane.svg')))
from sjepa.config import get_config, describe
from sjepa.models import build_model, pick_device
from sjepa.splits import fold_run_dir
cfg = get_config(); device = pick_device()
RUN_DIR = fold_run_dir(EXP_DIR, registry, cfg, FOLD)
model = build_model(cfg, device=device, repaired=True)
print(describe(cfg)); print('device:', device)
print('checkpoint directory:', RUN_DIR)

## Train only after splitting

The training helper checks the clip and source lists before making windows. It samples source videos uniformly, so sources with more windows do not dominate the updates. The usual budget is 800 updates; smoke mode uses 4 to check execution only. Stochastic masks let every joint appear as context and as a target.

In [ ]:
from sjepa.full_experiment import train_checkpoint
SMOKE = cfg.profile.endswith('smoke')
UPDATES = 4 if SMOKE else 800
state = train_checkpoint(model, train_recs, cfg, registry, FOLD, 'ssl',
                         UPDATES, device, RUN_DIR / 'ssl.pt')
print('training clips:', len(train_recs), '| training sources:', len({r.source_id for r in train_recs}))
print('final effective rank:', state.eff_rank[-1])

## Inspect training diagnostics

Loss alone cannot show that useful features were learned. Effective rank measures how many directions the features use; teacher drift measures how far teacher weights moved from student weights. These are training checks, not test accuracy or proof of gait learning.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
for a, values, title in zip(ax, [state.losses, state.eff_rank, state.teacher_drift],
                            ['training loss', 'effective rank', 'teacher drift']):
    a.plot(values); a.set_title(title); a.set_xlabel('update')
plt.tight_layout(); plt.show()
print('Saved a checkpoint tied to this cache, registry, fold, and configuration.')